# GPU-Accelerated Wavelet Pipeline Tutorial (Extended)

This tutorial demonstrates how to use the **PyTorch/CUDA** accelerated signal analysis pipeline to process multi-channel time series data.  
It builds upon the basic workflow (spectral similarity, candidate selection, continuous wavelet transform, coherence and propagation analysis) and adds deeper analyses such as **frequency‐band specific similarity** and **phase drift estimation**.  
Visualisation utilities for 2D and 3D plots are also showcased.  

> **Note**: This notebook assumes that a CUDA 12.x compatible GPU is available and that the required Python packages (`torch`, `numpy`, `scipy`, `matplotlib`, `plotly`, `streamlit` etc.) are installed.  If no GPU is detected, the computations will fall back to the CPU, albeit with longer runtimes.


## 1. Synthetic dataset

For demonstration purposes we generate a synthetic dataset consisting of four channels.  Each channel is a mixture of sine waves in different frequency bands plus some noise.  The sampling rate is set to **1 kHz** and the duration is **120 seconds**.  In the next sections we will explore how the GPU pipeline can handle datasets of this scale and larger.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Sampling parameters
fs = 1000.0  # Hz
T = 120.0    # seconds
n_samples = int(fs * T)
t = np.linspace(0, T, n_samples, endpoint=False)

# Define four synthetic signals with different frequency content and phase shifts
signals = np.zeros((4, n_samples), dtype=np.float32)

# Channel 0: low frequency + noise
signals[0] = 0.5 * np.sin(2*np.pi*2*t) + 0.2 * np.random.randn(n_samples)
# Channel 1: low + mid frequency, delayed
signals[1] = 0.5 * np.sin(2*np.pi*2*(t-0.1)) + 0.3 * np.sin(2*np.pi*10*(t-0.05)) + 0.2*np.random.randn(n_samples)
# Channel 2: mid + high frequency
signals[2] = 0.4 * np.sin(2*np.pi*20*t) + 0.3 * np.sin(2*np.pi*60*t) + 0.2 * np.random.randn(n_samples)
# Channel 3: high frequency, drifting phase
signals[3] = 0.4 * np.sin(2*np.pi*60*(t + 0.002*np.sin(2*np.pi*0.1*t))) + 0.2*np.random.randn(n_samples)

# Plot a short excerpt
duration = 1.0  # seconds
idx = slice(0, int(duration*fs))
plt.figure(figsize=(10, 4))
for i in range(4):
    plt.plot(t[idx], signals[i, idx] + i, label=f"Ch {i}")
plt.xlabel("Time (s)")
plt.title("Synthetic signals (first second)")
plt.legend()
plt.show()


## 2. Detrending and spectral analysis

Real recordings often exhibit slow drifts or linear trends.  We first **detrend** each channel using the GPU function `detrend_signals`.  Next we compute the **magnitude spectra** using the GPU accelerated FFT function and derive a **cosine similarity matrix** between channels.  High similarity indicates that two channels share similar frequency content.


In [ ]:
from signal_analysis_pipeline_torch import detrend_signals, compute_fft_magnitude, cosine_similarity_matrix, select_candidate_pairs

# Detrend signals on the GPU (returns NumPy array)
detrended = detrend_signals(signals)

# Compute FFT magnitudes
freqs_fft, mags = compute_fft_magnitude(detrended, sampling_rate=fs)

# Compute cosine similarity between magnitude spectra
sim_matrix = cosine_similarity_matrix(mags)

print("Cosine similarity matrix:
", np.round(sim_matrix, 2))

# Select candidate pairs above a threshold
threshold = 0.7
candidate_pairs = select_candidate_pairs(sim_matrix, threshold)
print("Candidate pairs (similarity >", threshold, "):", candidate_pairs)

## 3. Multi‑band similarity analysis

The basic spectral similarity uses the full bandwidth.  Often it is informative to restrict the analysis to specific frequency bands (e.g. **delta**, **theta**, **alpha**, **beta**, **gamma**).  The function `multi_band_similarity` applies a Butterworth band‑pass filter to each channel and computes two similarity metrics per band:

* **FFT cosine similarity** – compares the magnitude spectra of the filtered signals.
* **Time‑domain correlation** – Pearson correlation of the filtered time series.

Below we define a list of bands and compute these metrics.  The results are dictionaries mapping bands to their respective matrices.


In [ ]:
from signal_analysis_pipeline_torch import multi_band_similarity

# Define canonical EEG bands (in Hz)
bands = [(0.5, 4),   # delta
         (4, 8),    # theta
         (8, 12),   # alpha
         (12, 30),  # beta
         (30, 80)]  # gamma

# Compute multi‑band similarity
multi_sim = multi_band_similarity(detrended, sampling_rate=fs, bands=bands)

# Display one example band
band = (4, 8)  # theta
print(f"
Band {band} Hz - FFT cosine similarity:
", np.round(multi_sim[band]['fft_similarity'], 2))
print(f"
Band {band} Hz - Time‑domain correlation:
", np.round(multi_sim[band]['time_correlation'], 2))


## 4. Continuous wavelet transform and coherence

To capture time‑resolved relationships we compute the **continuous wavelet transform (CWT)** of each channel over a frequency range of interest.  The GPU implementation uses a bank of analytic Morlet filters applied via 1D convolutions.  We then compute the **wavelet coherence** between pairs of channels.  Coherence values range from 0 (no coupling) to 1 (perfect coupling) and are functions of both time and frequency.


In [ ]:
from signal_analysis_pipeline_torch import compute_cwt, wavelet_coherence

# Choose a candidate pair to analyse (e.g. first pair)
if candidate_pairs:
    ch_i, ch_j = candidate_pairs[0]
else:
    ch_i, ch_j = 0, 1

# Set CWT parameters
f_min = 1.0
f_max = 80.0
n_freqs = 64

# Compute CWT for the selected channels
freqs_cwt, coeffs_i = compute_cwt(detrended[ch_i], sampling_rate=fs, f_min=f_min, f_max=f_max, n_freqs=n_freqs)
_,              coeffs_j = compute_cwt(detrended[ch_j], sampling_rate=fs, f_min=f_min, f_max=f_max, n_freqs=n_freqs)

# Compute wavelet coherence and phase
coh, phase = wavelet_coherence(coeffs_i, coeffs_j)

print(f"Computed coherence shape: {coh.shape} (freqs x times)")


We can visualise the coherence spectrogram using a heatmap.  The y‑axis corresponds to frequency and the x‑axis to time.  Brighter colours indicate stronger coupling.  A large array (64×120,000) is processed efficiently on the GPU.


In [ ]:
import plotly.express as px

# Downsample time for visualisation (coherence is computed at 1 kHz)
step = int(fs / 50)  # keep ~50 fps on the x-axis
coh_ds = coh[:, ::step]
times_ds = np.arange(coh_ds.shape[1]) * (step / fs)

fig = px.imshow(
    coh_ds,
    aspect='auto',
    origin='lower',
    x=times_ds,
    y=freqs_cwt,
    color_continuous_scale='Viridis'
)
fig.update_layout(
    title=f'Wavelet coherence between channels {ch_i} and {ch_j}',
    xaxis_title='Time (s)',
    yaxis_title='Frequency (Hz)'
)
fig.show()


## 5. Propagation velocities and phase drift

From the wavelet phase differences we can estimate **propagation velocities** (or delays) by relating the phase to a time lag (`Δt = Δφ / 2πf`) and dividing the known sensor distance by this lag.  Moreover, by fitting a straight line to the unwrapped phase as a function of time we can obtain a **phase drift** per frequency.  A non‑zero drift indicates that the relative phase between two channels is changing systematically over time, which may correspond to slow changes in propagation speed or source dynamics.

We make use of the GPU‑accelerated `estimate_propagation_delays` and the new helper `phase_drift_analysis`.


In [ ]:
from signal_analysis_pipeline_torch import estimate_propagation_delays, phase_drift_analysis
from signal_analysis_pipeline import build_probe_positions

# Assume sensors are vertically aligned 1 cm apart (positions in cm)
layout = build_probe_positions([0.0, 1.0, 2.0, 3.0])
coords = layout.sensor_positions()

# Compute distance between selected pair in micrometres
import numpy as np

pos_i, pos_j = coords[ch_i], coords[ch_j]
dist_um = np.linalg.norm(pos_i - pos_j)

# Estimate velocities (µm/s) from phase and coherence
velocities = estimate_propagation_delays(phase, freqs_cwt, dist_um, sampling_rate=fs)

# Compute phase drift (s/s) per frequency
# The sampling interval is 1/fs seconds
dt = 1.0 / fs
drift = phase_drift_analysis(phase, freqs_cwt, dt)

# Plot drift versus frequency
plt.figure(figsize=(6, 4))
plt.plot(freqs_cwt, drift, marker='o')
plt.axhline(0, color='k', linestyle='--')
plt.xlabel('Frequency (Hz)')
plt.ylabel('Phase drift (s/s)')
plt.title('Phase drift between channels {} and {}'.format(ch_i, ch_j))
plt.show()


## 6. 3D sensor visualisation

Finally, we demonstrate how to plot the physical arrangement of the sensors in 3D using Plotly.  If a value is associated with each sensor (e.g. a mean velocity or drift magnitude), it can be encoded in the marker colour.  This is particularly helpful when examining large probe arrays.


In [ ]:
from visualization_tools import plot_sensor_positions_3d

# Example: colour sensors by mean propagation velocity magnitude
mean_vel = np.nanmean(np.abs(velocities), axis=1)

fig = plot_sensor_positions_3d(coords, values=mean_vel, labels=[f"Ch {i}" for i in range(coords.shape[0])])
fig


## 7. Saving results to a database

Large analyses can be persisted to a SQLite database for later inspection or sharing.  The function `save_results_to_db` (from `signal_analysis_pipeline` or its GPU counterpart) writes spectral features, similarity matrices, candidate pairs, coherence matrices and phase data into separate tables.  You can extend this functionality to include custom metrics such as the multi‑band similarities or drift estimates.

In practice, call:
```python
from signal_analysis_pipeline import save_results_to_db

save_results_to_db(
    'results.db',
    spectral_matrix=mags,
    similarity_matrix=sim_matrix,
    candidate_pairs=candidate_pairs,
    coherence_results={ (ch_i, ch_j): {'coherence': coh, 'phase': phase} },
    freqs_fft=freqs_fft,
    freqs_cwt={ (ch_i, ch_j): freqs_cwt },
    sensor_labels=[f"Ch {i}" for i in range(signals.shape[0])],
    layout=layout,
)
```
This will create the database file `results.db` in the working directory.  Use standard SQL queries (or pandas' `read_sql`) to explore the stored data.
